In [1]:
# ------ Module Import ------ #
import os
import pickle
import joblib
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

In [3]:
#Paths and seeds
RAND_SEED = 7
DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "Data")
RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), "Results")
PLOT_DIR = os.path.join(os.path.dirname(os.getcwd()), "Plots")
MODEL_DIR = os.path.join(os.path.dirname(os.getcwd()), "Models")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

In [4]:
#Data and metadata loading

df_100hvg = pd.read_pickle(os.path.join(DATA_DIR, "supervised_training_data_100hvg.pkl"))

with open(os.path.join(DATA_DIR, "supervised_data_100hvg_metadata.pkl"), "rb") as f:
    metadata_100hvg = pickle.load(f)

print(df_100hvg.shape)
df_100hvg.head()

(76893, 101)


,LYZ,HLA-DRA,S100A9,CD74,GNLY,NKG7,CTSS,S100A8,S100A4,CCL5,...,KLRD1,NPC2,NAP1L1,TSPO,ANXA2,DUSP6,LY6E,TKT,NOSIP,CellType
TCCCGATGTTAAGATG-1_7,4.764285,2.957022,5.596175,2.148582,-0.072270,-0.047454,4.362393,4.969619,3.909285,-0.640449,...,-0.090622,1.951062,0.308107,2.554441,0.789350,2.337746,0.828070,2.484555,1.105315,CD14 Mono
TTTCGATCACATTACG-1_1,4.147419,3.447765,3.366856,3.755852,0.005502,-0.040562,3.817119,2.211017,3.731036,0.063988,...,0.035326,2.260142,1.248083,1.380529,1.897829,2.360060,1.897827,2.581988,0.788886,CD14 Mono
AATGCCAGTGGACCAA-1_5,4.705282,2.494352,5.021750,2.587237,-0.030905,0.013300,4.060295,4.222051,2.973074,0.286722,...,-0.005076,1.169653,1.795462,1.560957,1.345841,2.453415,0.918939,2.257665,0.656441,CD14 Mono
CTTAGGAAGCCAACAG-1_8,4.741841,0.769762,5.458511,1.660862,0.656104,-0.062104,3.239459,4.821338,3.688125,0.423983,...,0.008045,0.842130,0.536949,2.514377,1.604810,0.564361,1.885159,1.595583,1.253817,CD14 Mono
GAATCACGTATTTCCT-1_13,5.081289,2.117163,5.781392,2.343518,-0.046718,0.114207,3.887106,5.049965,3.536385,0.008269,...,-0.031938,0.888212,1.286644,1.963565,1.736806,2.420097,0.394230,2.425130,-0.022348,CD14 Mono


In [5]:
metadata_100hvg

{'dataset_name': 'supervised_training_data_100hvg',
 'creation_date': '2026-08-07 14:54:45',
 'num_features': 100,
 'num_samples': 76893,
 'num_classes': 14,
 'feature_type': 'Highly Variable Genes (HVGs)',
 'hvg_selection_method': 'raw variance (top 100), validated via PCA/t-SNE/UMAP separability',
 'class_distribution': {'CD14 Mono': 9670,
  'CD4 TCM': 9250,
  'CD8 Naive': 9072,
  'CD8 TEM': 8636,
  'B naive': 7656,
  'CD4 Naive': 7114,
  'CD16 Mono': 6351,
  'NK': 6085,
  'B intermediate': 3483,
  'Dendritic cells': 2626,
  'CD8 TCM': 2400,
  'B memory': 2193,
  'CD4 TEM': 1335,
  'Treg': 1022}}

In [6]:
def prepare_supervised_dataset(
    df: pd.DataFrame,
    label_col: str = "CellType",
    meta_cols: list = ["CellType"],
    test_size: float = 0.15,
    val_size: float = 0.15,
    random_state: int = RAND_SEED
):
    print("Starting supervised dataset preparation...")

    expr_cols = [col for col in df.columns if col not in meta_cols]
    X = df[expr_cols].astype(np.float32)
    y_raw = df[label_col]

    print(f"Using {len(expr_cols)} genes in the dataset...")

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_raw)
    label_names = label_encoder.classes_
    print(f"Found {len(label_names)} unique cell types.")

    print("Splitting into train/val/test sets...")
    X_temp, X_test, y_temp, y_test, meta_temp, meta_test = train_test_split(
        X, y_encoded, df[meta_cols], test_size=test_size,
        stratify=y_encoded, random_state=random_state
    )
    val_fraction = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val, meta_train, meta_val = train_test_split(
        X_temp, y_temp, meta_temp, test_size=val_fraction,
        stratify=y_temp, random_state=random_state
    )

    print("Aligning common classes across splits (cuML compatibility)...")
    common_classes = np.intersect1d(np.unique(y_train), np.unique(y_val))
    mask_train = np.isin(y_train, common_classes)
    mask_val = np.isin(y_val, common_classes)
    mask_test = np.isin(y_test, common_classes)

    X_train, y_train, meta_train = X_train[mask_train], y_train[mask_train], meta_train.iloc[mask_train]
    X_val, y_val, meta_val = X_val[mask_val], y_val[mask_val], meta_val.iloc[mask_val]
    X_test, y_test, meta_test = X_test[mask_test], y_test[mask_test], meta_test.iloc[mask_test]

    class_weights_array = compute_class_weight("balanced", classes=np.unique(y_train), y=y_train)
    class_weights = dict(zip(np.unique(y_train), class_weights_array))

    print("\nFinal dataset sizes:")
    print(f"  Train: {X_train.shape[0]} samples")
    print(f"  Val:   {X_val.shape[0]} samples")
    print(f"  Test:  {X_test.shape[0]} samples")

    print("\nClass distribution in Train:")
    for cls, count in sorted(Counter(y_train).items()):
        print(f"  {label_names[cls]:<25}: {count} cells")

    print("\nClass Weights:")
    for cls, weight in class_weights.items():
        print(f"  {label_names[cls]:<25}: {weight:.2f}")

    return {
        "X_train": X_train, "X_val": X_val, "X_test": X_test,
        "y_train": y_train, "y_val": y_val, "y_test": y_test,
        "meta_train": meta_train, "meta_val": meta_val, "meta_test": meta_test,
        "label_encoder": label_encoder,
        "class_weights": class_weights,
        "feature_genes": list(expr_cols)
    }

In [7]:
spv_dataset = prepare_supervised_dataset(
    df_100hvg,
    label_col="CellType",
    meta_cols=["CellType"],
    test_size=0.15,
    val_size=0.15,
    random_state=RAND_SEED
)

joblib.dump(spv_dataset, os.path.join(DATA_DIR, "spv_split_dataset_100hvg.pkl"))
print(f"Saved dataset to {os.path.join(DATA_DIR, 'spv_split_dataset_100hvg.pkl')}")

Starting supervised dataset preparation...
Using 100 genes in the dataset...
Found 14 unique cell types.
Splitting into train/val/test sets...
Aligning common classes across splits (cuML compatibility)...

Final dataset sizes:
  Train: 53825 samples
  Val:   11534 samples
  Test:  11534 samples

Class distribution in Train:
  B intermediate           : 2438 cells
  B memory                 : 1535 cells
  B naive                  : 5360 cells
  CD14 Mono                : 6769 cells
  CD16 Mono                : 4445 cells
  CD4 Naive                : 4980 cells
  CD4 TCM                  : 6475 cells
  CD4 TEM                  : 935 cells
  CD8 Naive                : 6350 cells
  CD8 TCM                  : 1680 cells
  CD8 TEM                  : 6045 cells
  Dendritic cells          : 1838 cells
  NK                       : 4259 cells
  Treg                     : 716 cells

Class Weights:
  B intermediate           : 1.58
  B memory                 : 2.50
  B naive                  : 0.7

In [8]:
spv_dataset = joblib.load(os.path.join(DATA_DIR, "spv_split_dataset_100hvg.pkl"))

def verify_cell_types_integrity(spv_dataset):
    print("Verifying cell types integrity...")

    for split in ["train", "val", "test"]:
        y_split = spv_dataset[f"y_{split}"]
        unique_classes = np.unique(y_split)
        print(f"Unique classes in {split} set: {unique_classes}")

    train_classes = set(np.unique(spv_dataset["y_train"]))
    val_classes = set(np.unique(spv_dataset["y_val"]))
    test_classes = set(np.unique(spv_dataset["y_test"]))

    common_classes = train_classes.intersection(val_classes, test_classes)
    print(f"Common classes across all splits: {sorted(list(common_classes))}")

    label_encoder = spv_dataset["label_encoder"]
    class_names = label_encoder.classes_

    print("\nClass distribution ratios (% of total):")
    print(f"{'Class':<15} {'Train %':<10} {'Val %':<10} {'Test %':<10} {'Total %':<10}")
    print("-" * 55)

    total_train = len(spv_dataset["y_train"])
    total_val = len(spv_dataset["y_val"])
    total_test = len(spv_dataset["y_test"])
    total_all = total_train + total_val + total_test

    for i, class_name in enumerate(class_names):
        train_count = np.sum(spv_dataset["y_train"] == i)
        val_count = np.sum(spv_dataset["y_val"] == i)
        test_count = np.sum(spv_dataset["y_test"] == i)
        total_count = train_count + val_count + test_count

        train_ratio = train_count / total_train * 100
        val_ratio = val_count / total_val * 100
        test_ratio = test_count / total_test * 100
        total_ratio = total_count / total_all * 100

        print(f"{class_name:<15} {train_ratio:>8.2f}% {val_ratio:>8.2f}% {test_ratio:>8.2f}% {total_ratio:>8.2f}%")

    print("\nDataset split ratios:")
    print(f"Train: {total_train / total_all * 100:.2f}% ({total_train} samples)")
    print(f"Val:   {total_val / total_all * 100:.2f}% ({total_val} samples)")
    print(f"Test:  {total_test / total_all * 100:.2f}% ({total_test} samples)")

    print("\nPotential CV issues assessment:")
    if len(train_classes) != len(class_names) or len(val_classes) != len(class_names) or len(test_classes) != len(class_names):
        print("Warning: Some classes are missing from one or more splits!")
    else:
        print("✓ All classes are represented in each split")

    class_balance_issues = []
    for i, class_name in enumerate(class_names):
        train_count = np.sum(spv_dataset["y_train"] == i)
        val_count = np.sum(spv_dataset["y_val"] == i)
        test_count = np.sum(spv_dataset["y_test"] == i)
        if train_count < 5 or val_count < 5 or test_count < 5:
            class_balance_issues.append(f"{class_name} (train={train_count}, val={val_count}, test={test_count})")

    if class_balance_issues:
        print("Warning: Some classes have very few samples in some splits:")
        for issue in class_balance_issues:
            print(f"  - {issue}")
    else:
        print("✓ All classes have sufficient samples in each split")

    print("\nIntegrity verification complete!")

In [9]:
verify_cell_types_integrity(spv_dataset)

Verifying cell types integrity...
Unique classes in train set: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Unique classes in val set: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Unique classes in test set: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13]
Common classes across all splits: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13)]

Class distribution ratios (% of total):
Class           Train %    Val %      Test %     Total %   
-------------------------------------------------------
B intermediate      4.53%     4.53%     4.53%     4.53%
B memory            2.85%     2.85%     2.85%     2.85%
B naive             9.96%     9.95%     9.95%     9.96%
CD14 Mono          12.58%    12.57%    12.58%    12.58%
CD16 Mono           8.26%     8.26%     8.26%     8.26%
CD4 Naive           9.25%     9.25%     9.25%     9.25%
CD4 TCM            12.03%    12.03% 

In [10]:
def print_per_class_split_ratios(spv_dataset):
    """
    For each cell type, show what percentage of that type's own cells
    landed in train/val/test — a per-class complement to the earlier
    dataset-wide distribution check.
    """
    label_encoder = spv_dataset["label_encoder"]
    class_names = label_encoder.classes_

    print("Split ratios per class:")
    print(f"{'Class':<15} {'Train %':<10} {'Val %':<10} {'Test %':<10}")
    print("-" * 45)

    for i, class_name in enumerate(class_names):
        train_count = np.sum(spv_dataset["y_train"] == i)
        val_count = np.sum(spv_dataset["y_val"] == i)
        test_count = np.sum(spv_dataset["y_test"] == i)
        class_total = train_count + val_count + test_count

        train_ratio = train_count / class_total * 100
        val_ratio = val_count / class_total * 100
        test_ratio = test_count / class_total * 100

        print(f"{class_name:<15} {train_ratio:>8.2f}% {val_ratio:>8.2f}% {test_ratio:>8.2f}%")

In [11]:
print_per_class_split_ratios(spv_dataset)

Split ratios per class:
Class           Train %    Val %      Test %    
---------------------------------------------
B intermediate     70.00%    15.02%    14.99%
B memory           70.00%    15.00%    15.00%
B naive            70.01%    14.99%    14.99%
CD14 Mono          70.00%    14.99%    15.01%
CD16 Mono          69.99%    15.01%    15.01%
CD4 Naive          70.00%    15.00%    15.00%
CD4 TCM            70.00%    14.99%    15.01%
CD4 TEM            70.04%    14.98%    14.98%
CD8 Naive          70.00%    15.00%    15.00%
CD8 TCM            70.00%    15.00%    15.00%
CD8 TEM            70.00%    15.01%    15.00%
Dendritic cells    69.99%    15.00%    15.00%
NK                 69.99%    15.00%    15.00%
Treg               70.06%    14.97%    14.97%


**Note:** this per-class split-ratio check is implemented as a standalone function rather than folded into `verify_cell_types_integrity()`. It answers a distinct question from the earlier dataset-wide distribution table: *within a given cell type's own cells, what fraction landed in each split*; as opposed to *what fraction of the whole dataset each class-split combination represents*. Keeping it separate makes each function single-purpose and independently reusable (e.g., if the earlier checks pass but you want to re-verify only this specific property after some other change).